# Rail-V vibration / acoustic training

Runs the pipeline in `Rail-V/vibration` on the data you downloaded to Google Drive.

**Before starting:** Runtime > Change runtime type > **T4 GPU** (needed for step 6 only).

Steps: mount Drive, get the code, inspect the data, build the manifest and splits, run the random-forest baseline, fine-tune AST.

## 1. Mount Drive and set paths
Change `DATA` if you downloaded somewhere else. Results go to `RUNS` on Drive so they survive the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/rail-v_dataset'   # folder download.py --out wrote into
RUNS = '/content/drive/MyDrive/rail-v-runs'
import os; os.makedirs(RUNS, exist_ok=True)

## 2. Get the code
Use `main` once the pipeline PR is merged. If the repo is private, put a GitHub token in `TOKEN` (Colab: key icon > add secret `GITHUB_TOKEN`).

In [ ]:
BRANCH = 'main'
TOKEN = ''
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
auth = f'{TOKEN}@' if TOKEN else ''
!rm -rf /content/Research-Projects
!git clone -q -b $BRANCH https://{auth}github.com/Zcodeod-OG/Research-Projects.git /content/Research-Projects
%cd /content/Research-Projects/Rail-V/vibration
!pip -q install -r requirements.txt

## 3. Inspect the data
Copy this cell's output into the project thread. It shows the folder layout and the UPM corrugation `.mat` variables, which the corrugation loader still needs.

In [ ]:
!python inspect_data.py $DATA

## 3b. Cut the corrugation runs into labelled clips
Once only. Labels come from the two corrugated sections reported in the dataset's paper (km 4.855-4.914 and 5.239-5.326). Writes 1 s WAV clips next to the `.mat` files.

In [ ]:
!python prepare_corrugation.py --data $DATA

## 4. Build the manifest and grouped splits
Writes one row per recording and fixes train / val / test once. Check the class table it prints: each class should have roughly 240 clips.

In [ ]:
!python build_manifest.py --data $DATA --out $RUNS/manifest.csv

## 5. Baseline: MFCC + random forest
CPU only, a few minutes. This is the number the deep model has to beat.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --out $RUNS/baseline_rf --cv

## 6. Fine-tune AST (GPU)
Downloads the AudioSet-pretrained AST, fine-tunes on 5 s windows and keeps the best checkpoint on Drive. Expect a few minutes per epoch on a T4; loading the clips from Drive takes a few minutes first.

In [ ]:
!python train_ast.py --manifest $RUNS/manifest.csv --out $RUNS/ast

## 6b. Corrugation: corrugated vs normal track
A separate two-class task per sensor, trained on the 40/50 km/h runs, validated on 60 km/h and tested on 75 km/h. The axlebox accelerometer is usually the stronger signal for corrugation.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --source corrugation_axle --fs 4000 --cv --out $RUNS/corr_axle_rf
!python baseline_rf.py --manifest $RUNS/manifest.csv --source corrugation_mic --cv --out $RUNS/corr_mic_rf
!python train_ast.py --manifest $RUNS/manifest.csv --source corrugation_mic --window 1 --hop 0.5 --out $RUNS/corr_mic_ast

## 7. Compare
Clip-level macro-F1 on the same held-out test split.

In [ ]:
import json
for name in ['baseline_rf', 'ast', 'corr_axle_rf', 'corr_mic_rf', 'corr_mic_ast']:
    p = f'{RUNS}/{name}/results.json'
    if os.path.exists(p):
        t = json.load(open(p))['test']['clip']
        print(f"{name:12} clip acc {t['accuracy']}  macro-F1 {t['macro_f1']}  recall {t['recall']}")